## Libraries

In [27]:
import pandas as pd
import requests
import zipfile
import os

# Getting data
from io import StringIO

# Splitting
from sklearn.model_selection import cross_val_score, train_test_split

# Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

## Functions

### Get Data

In [7]:
def get_data():

    dfResult = None

    # Download German Credit Data from UCI ML Repository
    url = "https://archive.ics.uci.edu/static/public/144/statlog+german+credit+data.zip"

    # Download the zip file
    response = requests.get(url)
    response.raise_for_status()

    # Extract the zip file and load data into DataFrame
    from io import BytesIO
    with zipfile.ZipFile(BytesIO(response.content)) as zip_file:
        # List files in the zip
        file_list = zip_file.namelist()
        print("Files in the dataset:")
        for file in file_list:
            print(f"  {file}")
        
        # Find the main data file (usually german.data)
        data_file = None
        for file in file_list:
            if file.endswith('.data') or 'german' in file.lower():
                data_file = file
                break
        
        if data_file:
            with zip_file.open(data_file) as f:
                # Read the data file
                data_content = f.read().decode('utf-8')
                
                # Load into pandas DataFrame
                # German credit data is space-separated
                dfResult = pd.read_csv(StringIO(data_content), sep=' ', header=None)
                
                print(f"\nDataset loaded successfully!")
        else:
            print("Could not find the main data file in the zip archive")
        
    return dfResult


### Data cleaning

In [10]:
def add_column_names(df):
    """
    Adds proper column names to the dataframe
    
    Args:
        df (pd.DataFrame): The dataframe to add column names to
    
    Returns:
        pd.DataFrame: Dataframe with proper column names
    """
    column_names = [
        'checking_account_status',    # Attribute 1
        'duration_months',            # Attribute 2
        'credit_history',            # Attribute 3
        'purpose',                   # Attribute 4
        'credit_amount',             # Attribute 5
        'savings_account',           # Attribute 6
        'employment_since',          # Attribute 7
        'installment_rate',          # Attribute 8
        'personal_status_sex',       # Attribute 9
        'other_debtors',             # Attribute 10
        'residence_since',           # Attribute 11
        'property',                  # Attribute 12
        'age_years',                 # Attribute 13
        'other_installment_plans',   # Attribute 14
        'housing',                   # Attribute 15
        'existing_credits',          # Attribute 16
        'job',                       # Attribute 17
        'num_dependents',            # Attribute 18
        'telephone',                 # Attribute 19
        'foreign_worker',            # Attribute 20
        'class'                      # Target variable (1=good, 2=bad)
    ]
    
    df_named = df.copy()
    df_named.columns = column_names
    
    return df_named



In [9]:
def get_feature_mappings():
    """Returns a dictionary containing all feature mappings"""
    
    mappings = {
        # Attribute 1: Status of existing checking account
        0: {
            'A11': '< 0 DM',
            'A12': '0 <= ... < 200 DM', 
            'A13': '>= 200 DM / salary assignments for at least 1 year',
            'A14': 'no checking account'
        },
        
        # Attribute 3: Credit history
        2: {
            'A30': 'no credits taken / all credits paid back duly',
            'A31': 'all credits at this bank paid back duly',
            'A32': 'existing credits paid back duly till now',
            'A33': 'delay in paying off in the past',
            'A34': 'critical account / other credits existing (not at this bank)'
        },
        
        # Attribute 4: Purpose
        3: {
            'A40': 'car (new)',
            'A41': 'car (used)',
            'A42': 'furniture/equipment',
            'A43': 'radio/television',
            'A44': 'domestic appliances',
            'A45': 'repairs',
            'A46': 'education',
            'A48': 'retraining',
            'A49': 'business',
            'A410': 'others'
        },
        
        # Attribute 6: Savings account/bonds
        5: {
            'A61': '< 100 DM',
            'A62': '100 <= ... < 500 DM',
            'A63': '500 <= ... < 1000 DM',
            'A64': '>= 1000 DM',
            'A65': 'unknown / no savings account'
        },
        
        # Attribute 7: Present employment since
        6: {
            'A71': 'unemployed',
            'A72': '< 1 year',
            'A73': '1 <= ... < 4 years',
            'A74': '4 <= ... < 7 years',
            'A75': '>= 7 years'
        },
        
        # Attribute 9: Personal status and sex
        8: {
            'A91': 'male : divorced/separated',
            'A92': 'female : divorced/separated/married',
            'A93': 'male : single',
            'A94': 'male : married/widowed',
            'A95': 'female : single'
        },
        
        # Attribute 10: Other debtors / guarantors
        9: {
            'A101': 'none',
            'A102': 'co-applicant',
            'A103': 'guarantor'
        },
        
        # Attribute 12: Property
        11: {
            'A121': 'real estate',
            'A122': 'building society savings agreement / life insurance',
            'A123': 'car or other, not in attribute 6',
            'A124': 'unknown / no property'
        },
        
        # Attribute 14: Other installment plans
        13: {
            'A141': 'bank',
            'A142': 'stores',
            'A143': 'none'
        },
        
        # Attribute 15: Housing
        14: {
            'A151': 'rent',
            'A152': 'own',
            'A153': 'for free'
        },
        
        # Attribute 17: Job
        16: {
            'A171': 'unemployed / unskilled - non-resident',
            'A172': 'unskilled - resident',
            'A173': 'skilled employee / official',
            'A174': 'management / self-employed / highly qualified employee / officer'
        },
        
        # Attribute 19: Telephone
        18: {
            'A191': 'none',
            'A192': 'yes, registered under the customers name'
        },
        
        # Attribute 20: Foreign worker
        19: {
            'A201': 'yes',
            'A202': 'no'
        }
    }
    
    return mappings

In [11]:
def transform_dataframe_with_labels(df):
    """
    Transforms the entire dataframe by replacing categorical codes with actual meanings
    
    Args:
        df (pd.DataFrame): The original dataframe with categorical codes
    
    Returns:
        pd.DataFrame: A new dataframe with readable labels
    """
    # Create a copy of the dataframe
    df_transformed = df.copy()
    
    # Get the mappings
    mappings = get_feature_mappings()
    
    # Apply mappings to categorical columns
    for column_index, mapping_dict in mappings.items():
        if column_index < len(df.columns):
            df_transformed.iloc[:, column_index] = df_transformed.iloc[:, column_index].map(mapping_dict).fillna(df_transformed.iloc[:, column_index])
    
    return df_transformed

## Extracting Data & Cleaning

First we download the dataset from the URL https://archive.ics.uci.edu/static/public/144/statlog+german+credit+data.zip

In [12]:

df_credit = get_data()
df_credit.head()

Files in the dataset:
  german.data
  german.data-numeric
  german.doc
  Index

Dataset loaded successfully!


,0,1,2,3,4,5,6,7,8,9,...,11,12,13,14,15,16,17,18,19,20
0,A11,6,A34,A43,1169,A65,A75,4,A93,A101,...,A121,67,A143,A152,2,A173,1,A192,A201,1
1,A12,48,A32,A43,5951,A61,A73,2,A92,A101,...,A121,22,A143,A152,1,A173,1,A191,A201,2
2,A14,12,A34,A46,2096,A61,A74,2,A93,A101,...,A121,49,A143,A152,1,A172,2,A191,A201,1
3,A11,42,A32,A42,7882,A61,A74,2,A93,A103,...,A122,45,A143,A153,1,A173,2,A191,A201,1
4,A11,24,A33,A40,4870,A61,A73,3,A93,A101,...,A124,53,A143,A153,2,A173,2,A191,A201,2


Now we clean the data. We add the column names and the real values of the categorical features.

In [15]:
df_credit = add_column_names(df_credit)
df_credit = transform_dataframe_with_labels(df_credit)
df_credit.head(10)

,checking_account_status,duration_months,credit_history,purpose,credit_amount,savings_account,employment_since,installment_rate,personal_status_sex,other_debtors,...,property,age_years,other_installment_plans,housing,existing_credits,job,num_dependents,telephone,foreign_worker,class
0,< 0 DM,6,critical account / other credits existing (not...,radio/television,1169,unknown / no savings account,>= 7 years,4,male : single,none,...,real estate,67,none,own,2,skilled employee / official,1,"yes, registered under the customers name",yes,1
1,0 <= ... < 200 DM,48,existing credits paid back duly till now,radio/television,5951,< 100 DM,1 <= ... < 4 years,2,female : divorced/separated/married,none,...,real estate,22,none,own,1,skilled employee / official,1,none,yes,2
2,no checking account,12,critical account / other credits existing (not...,education,2096,< 100 DM,4 <= ... < 7 years,2,male : single,none,...,real estate,49,none,own,1,unskilled - resident,2,none,yes,1
3,< 0 DM,42,existing credits paid back duly till now,furniture/equipment,7882,< 100 DM,4 <= ... < 7 years,2,male : single,guarantor,...,building society savings agreement / life insu...,45,none,for free,1,skilled employee / official,2,none,yes,1
4,< 0 DM,24,delay in paying off in the past,car (new),4870,< 100 DM,1 <= ... < 4 years,3,male : single,none,...,unknown / no property,53,none,for free,2,skilled employee / official,2,none,yes,2
5,no checking account,36,existing credits paid back duly till now,education,9055,unknown / no savings account,1 <= ... < 4 years,2,male : single,none,...,unknown / no property,35,none,for free,1,unskilled - resident,2,"yes, registered under the customers name",yes,1
6,no checking account,24,existing credits paid back duly till now,furniture/equipment,2835,500 <= ... < 1000 DM,>= 7 years,3,male : single,none,...,building society savings agreement / life insu...,53,none,own,1,skilled employee / official,1,none,yes,1
7,0 <= ... < 200 DM,36,existing credits paid back duly till now,car (used),6948,< 100 DM,1 <= ... < 4 years,2,male : single,none,...,"car or other, not in attribute 6",35,none,rent,1,management / self-employed / highly qualified ...,1,"yes, registered under the customers name",yes,1
8,no checking account,12,existing credits paid back duly till now,radio/television,3059,>= 1000 DM,4 <= ... < 7 years,2,male : divorced/separated,none,...,real estate,61,none,own,1,unskilled - resident,1,none,yes,1
9,0 <= ... < 200 DM,30,critical account / other credits existing (not...,car (new),5234,< 100 DM,unemployed,4,male : married/widowed,none,...,"car or other, not in attribute 6",28,none,own,2,management / self-employed / highly qualified ...,1,none,yes,2


In [26]:
df_credit.info()

<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 21 columns):
 #   Column                   Non-Null Count  Dtype
---  ------                   --------------  -----
 0   checking_account_status  1000 non-null   str  
 1   duration_months          1000 non-null   int64
 2   credit_history           1000 non-null   str  
 3   purpose                  1000 non-null   str  
 4   credit_amount            1000 non-null   int64
 5   savings_account          1000 non-null   str  
 6   employment_since         1000 non-null   str  
 7   installment_rate         1000 non-null   int64
 8   personal_status_sex      1000 non-null   str  
 9   other_debtors            1000 non-null   str  
 10  residence_since          1000 non-null   int64
 11  property                 1000 non-null   str  
 12  age_years                1000 non-null   int64
 13  other_installment_plans  1000 non-null   str  
 14  housing                  1000 non-null   str  
 15  existing_credits

## Splitting into Training, Test and Validation

In [25]:
df_train_val, df_test = train_test_split(df_credit, test_size=0.15, random_state=1505)

# Validation set (15% of total data) -> 0.15 / (1 - 0.15) ≈ 0.176
df_train, df_val = train_test_split(df_train_val, test_size=0.176, random_state=1505)

# Print the sizes of the datasets
print(f"Training set size: {len(df_train)}")
print(f"Validation set size: {len(df_val)}")
print(f"Test set size: {len(df_test)}")

df_train.head(3)

Training set size: 700
Validation set size: 150
Test set size: 150


,checking_account_status,duration_months,credit_history,purpose,credit_amount,savings_account,employment_since,installment_rate,personal_status_sex,other_debtors,...,property,age_years,other_installment_plans,housing,existing_credits,job,num_dependents,telephone,foreign_worker,class
797,no checking account,12,critical account / other credits existing (not...,furniture/equipment,1258,< 100 DM,< 1 year,2,female : divorced/separated/married,none,...,building society savings agreement / life insu...,22,none,rent,2,unskilled - resident,1,none,yes,1
832,< 0 DM,45,no credits taken / all credits paid back duly,business,11816,< 100 DM,>= 7 years,2,male : single,none,...,"car or other, not in attribute 6",29,none,rent,2,skilled employee / official,1,none,yes,2
358,no checking account,12,existing credits paid back duly till now,radio/television,776,< 100 DM,1 <= ... < 4 years,4,male : married/widowed,none,...,real estate,28,none,own,1,skilled employee / official,1,none,yes,1


## Model Pipeline

In [37]:
categorical_features = ['checking_account_status', 'credit_history', 'purpose', 'savings_account', 'employment_since', 'personal_status_sex', 'other_debtors', 'property', 'other_installment_plans', 'housing', 'job', 'telephone', 'foreign_worker']

numerical_features = ['duration_months', 'credit_amount', 'installment_rate', 'residence_since', 'age_years', 'existing_credits', 'num_dependents']


pipe_categorical = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

pipe_numerical = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='mean')),
    ('scaler', StandardScaler())
])

preprocessor = ColumnTransformer(
    transformers=[
        ('categorical', pipe_categorical, categorical_features),
        ('numerical', pipe_numerical, numerical_features)
    ]
)

preprocessor.set_output(transform='pandas')
df_train_preprocessed = preprocessor.fit_transform(df_train)
df_train_preprocessed.head(5)

,categorical__checking_account_status_0 <= ... < 200 DM,categorical__checking_account_status_< 0 DM,categorical__checking_account_status_>= 200 DM / salary assignments for at least 1 year,categorical__checking_account_status_no checking account,categorical__credit_history_all credits at this bank paid back duly,categorical__credit_history_critical account / other credits existing (not at this bank),categorical__credit_history_delay in paying off in the past,categorical__credit_history_existing credits paid back duly till now,categorical__credit_history_no credits taken / all credits paid back duly,categorical__purpose_business,...,"categorical__telephone_yes, registered under the customers name",categorical__foreign_worker_no,categorical__foreign_worker_yes,numerical__duration_months,numerical__credit_amount,numerical__installment_rate,numerical__residence_since,numerical__age_years,numerical__existing_credits,numerical__num_dependents
797,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,-0.740132,-0.713267,-0.873157,1.035222,-1.210534,1.011876,-0.42478
832,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,...,0.0,0.0,1.0,1.920474,3.041838,-0.873157,1.035222,-0.603035,1.011876,-0.42478
358,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,...,0.0,0.0,1.0,-0.740132,-0.884698,0.919248,-0.778684,-0.689821,-0.711514,-0.42478
572,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,...,1.0,0.0,1.0,0.227361,0.771632,-0.873157,1.035222,-0.863392,-0.711514,-0.42478
896,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,...,1.0,0.0,1.0,-0.014512,-0.233832,0.919248,1.035222,-0.689821,-0.711514,-0.42478
